# Combinatorial auction

This notebook shows how combRUM can also be used to find equilibrium prices and an allocation for a combinatorial auction.


## Primitives

There are $N$ bidders and $M$ indivisible items. Each bidder's preferences are generated by an assignment problem introduced below.


In [11]:
import numpy as np

rng = np.random.default_rng(123)

N = 50  # agents
M = 100  # items
L = 20  # tasks

# R[i, j, l] is agent i's surplus from assigning item j to task l.
R = rng.uniform(0.0, 1.0, size=(N, M, L)) 


## Assignment Valuations

Bidder $i$ has $L$ tasks. The tensor $R_{ij\ell}$ stores bidder $i$'s surplus from assigning item $j$ to task $\ell$. For bundle $B$,

$$U_i(B)=\max_{\alpha}\sum_{j\in B}\sum_{\ell=1}^L r_{ij\ell}\alpha_{j\ell},$$

where $\alpha$ varies over all possible assignments of elements in $B$ to tasks in $[L]$. These are assignment valuations in the sense of Hatfield and Milgrom (2005), and they satisfy gross substitutes.

The demand routine below uses greedy local search. For gross-substitute valuations, this local search is exact. See Gul and Stacchetti (1999).


In [12]:
from scipy.optimize import linear_sum_assignment

def U(i, B):
    values = R[i, B]
    rows, cols = linear_sum_assignment(values, maximize=True)
    return values[rows, cols].sum()

def demand_and_value(i, prices):
    bundle = np.zeros(M, dtype=bool)
    value = 0.0
    paid = 0.0

    while True:
        best_gain = 0.0
        best_item = None
        for item in range(M):
            if bundle[item]:
                continue
            bundle[item] = True
            gain = U(i, bundle) - value - prices[item]
            bundle[item] = False
            if gain > best_gain:
                best_gain = gain
                best_item = item

        if best_item is None:
            return bundle, value - paid
        bundle[best_item] = True
        value += best_gain + prices[best_item]
        paid += prices[best_item]


## Equilibrium and LP Relaxation

A Walrasian equilibrium with indivisible items is an allocation $(B_i)_{i\in[N]}$ and prices $p\ge 0$ such that each bidder receives a demanded bundle,

$$B_i\in\arg\max_{B\subseteq [M]}\{U_i(B)-p(B)\},$$

and the allocation clears the item market. Each item has one unit of supply.

The integer welfare problem has one variable $\pi_{iB}$ for bidder $i$ receiving bundle $B$. Its LP relaxation is

$$\begin{aligned}
\max_{\pi\ge 0}\quad & \sum_{i\in[N]}\sum_{B\subseteq[M]} U_i(B)\pi_{iB}\\
\text{s.t.}\quad & \sum_{B\subseteq[M]} \pi_{iB}\le 1 && \forall i\in[N],\\
& \sum_{i\in[N]}\sum_{B\ni j}\pi_{iB}\le 1 && \forall j\in[M].
\end{aligned}$$

The dual is

$$\begin{aligned}
\min_{u,p\ge 0}\quad & \sum_{i\in[N]} u_i+\sum_{j\in[M]} p_j\\
\text{s.t.}\quad & u_i+\sum_{j\in B}p_j\ge U_i(B) && \forall i\in[N],\ B\subseteq[M].
\end{aligned}$$

Equivalently, $u_i\ge U_i(B)-p(B)$. For gross-substitute valuations, the LP relaxation is tight.


## combRUM objects

In this example, the unknown "parameters" are the item prices. The feature map turns a bidder's bundle into the corresponding price term, and the oracle returns the bundle demanded at the current prices. We use the observed bundles only to record that each item has one unit of supply (any assignment of items to bidders would work here).


In [14]:
import combrum as cb

parameters = cb.Parameters({"price": (0.0, 100.0, M)})

item_owner = rng.integers(N, size=M)

observed_bundles = np.zeros((N, M), dtype=bool)
observed_bundles[item_owner, np.arange(M)] = True


In [15]:
class AuctionFeatures(cb.FeatureMap):
    def features_batch(self, ids, bundles):
        price_coefficients = -1.0 * bundles
        assignment_values = np.array([U(i, B) for i, B in zip(ids, bundles)])
        return price_coefficients, assignment_values


In [16]:
class AuctionOracle(cb.Oracle):
    def price_batch(self, prices, ids):
        bundles, values = zip(*(demand_and_value(i, prices) for i in ids))
        return cb.DemandBatch.exact(ids, np.array(bundles), np.array(values))


In [17]:
model = cb.Model(
    AuctionOracle(),
    parameters,
    features=AuctionFeatures(),
    formulation=cb.NSlack,
)

data = cb.Data(
    observed_bundles=observed_bundles,
    shocks=np.zeros((N, 1)),
    observables=np.arange(N),
)


## Prices

Row generation solves the dual LP without enumerating all bundles. At each price vector, the oracle finds a demanded bundle and adds the corresponding dual constraint when it is violated.


In [18]:
result = cb.estimate(
    model,
    data,
    master_backend="highs",
    return_cut_duals=True,
    activity=cb.ActivityConfig(label="auction", level="iterations", stdout=True),
)
eq_prices = result.parameters.unpack(result.theta_hat)["price"]

[auction] row generation: N=50 S=1 K=100 agents=50 ranks=1 tol=1.000e-06 max_iter=1000
[auction] iter         gap        dgap         obj        dobj  +cuts   cuts  priced  inexact   price  master      dt    total
[auction]    0   1.987e+01           -   8.182e+01           -     50     50      50        0   0.30s   0.00s   0.30s    0.30s
[auction]    1   1.979e+01  -7.754e-02   9.017e+01   8.356e+00     50    100      50        0   0.30s   0.00s   0.30s    0.60s
[auction]    2   1.967e+01  -1.245e-01   9.618e+01   6.010e+00     50    150      50        0   0.31s   0.00s   0.31s    0.91s
[auction]    3   1.921e+01  -4.622e-01   9.844e+01   2.255e+00     50    200      50        0   0.31s   0.00s   0.31s    1.22s
[auction]    4   1.225e+01  -6.961e+00   9.859e+01   1.522e-01     50    250      50        0   0.34s   0.00s   0.34s    1.56s
[auction]    5   1.549e+00  -1.070e+01   9.864e+01   5.343e-02     50    300      50        0   0.28s   0.00s   0.29s    1.85s
[auction]    6   1.939e+

## Allocation

The dual variables on the generated constraints recover the allocation weights $\pi_{iB}$. Because assignment valuations satisfy gross substitutes, the LP relaxation is tight. In this run row generation returns an integral allocation: each bidder has at most one bundle with positive dual variable at the solution.


In [19]:
dual = result.cut_duals
dual_bundles = dual.bundle_table[dual.bundle_row_ids]

assert all((dual.pis == 0) | (dual.pis == 1))

matrix_allocation = np.zeros((N, M), dtype=bool)
active = dual.pis == 1
matrix_allocation[dual.agent_ids[active]] = dual_bundles[active]

## Check

The allocation should clear the market, and no bidder should prefer its demanded bundle at the computed prices.


In [20]:
# market clearing
mrk_clears = all(matrix_allocation.sum(0) == 1)

regret = np.zeros(N)
for i in range(N):
    _, demand_value = demand_and_value(i, eq_prices)
    allocation_value = U(i, matrix_allocation[i]) - eq_prices @ matrix_allocation[i]
    regret[i] = demand_value - allocation_value
max_regret = regret.max()

print("market clears:", mrk_clears)
print("bidders optimality:", max_regret)

market clears: True
bidders optimality: 8.881784197001252e-16


## References

Gul, Faruk, and Ennio Stacchetti. 1999. "Walrasian Equilibrium with Gross Substitutes." *Journal of Economic Theory* 87(1): 95-124.

Hatfield, John William, and Paul R. Milgrom. 2005. "Matching with Contracts." *American Economic Review* 95(4): 913-935.
